# 细粒度 Capability 评估

本 notebook 会：
1. 读取 `meta.json` 中的 capability label（`capability_assignments`）。
2. 读取标准答案和模型抽取结果。
3. 复用 `test_dataset_evaluate.py` 里的 `Doc2DBEvaluator.calculate_cell_metrics` 做逐 cell 比对。
4. 按 capability 统计 `precision / recall / f1`。


In [ ]:
import json
from pathlib import Path
from collections import defaultdict

import pandas as pd

from test_dataset_evaluate import Doc2DBEvaluator

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)


In [ ]:
# ====== 路径配置（按需修改） ======
LABEL_SOURCE_PATH = Path("dataset/case/base_latest/education/case1/meta.json")
ANSWER_DIR = Path("dataset/case/base_latest/education/case1/tables")
SCHEMA_PATH = Path("dataset/case/base_latest/education/case1/schema.json")
GENERATED_DATA_PATH = Path("dataset/case/base_latest_output/llm/education/case1/education_case1_gpt_4o_only/extracted_data.json")

assert LABEL_SOURCE_PATH.exists(), f"label来源不存在: {LABEL_SOURCE_PATH}"
assert ANSWER_DIR.exists(), f"answer目录不存在: {ANSWER_DIR}"
assert SCHEMA_PATH.exists(), f"schema不存在: {SCHEMA_PATH}"
assert GENERATED_DATA_PATH.exists(), f"generated结果不存在: {GENERATED_DATA_PATH}"


In [ ]:
# ====== relation-level 统计（整目录汇总）======
import json
from pathlib import Path
from collections import Counter

# 用户指定目录（若不存在则回退到常见目录）
CASE_ROOT = Path("dataset/case/base_latest")

meta_files = sorted([p for p in CASE_ROOT.rglob("meta.json") if p.is_file()])
print(f"扫描目录: {CASE_ROOT}")
print(f"发现 meta.json 数量: {len(meta_files)}")

all_type_counter = Counter()
relation_rows = []

for meta_path in meta_files:
    with open(meta_path, "r", encoding="utf-8") as fp:
        meta = json.load(fp)

    type_map = meta.get("type", {}) or {}
    if not isinstance(type_map, dict):
        continue

    all_type_counter.update(str(v).strip() for v in type_map.values())

    parts = meta_path.parts
    # 兼容 .../<domain>/<case>/meta.json 或 .../llm/<domain>/<case>/.../meta.json
    domain = parts[-3] if len(parts) >= 3 else "unknown"
    case_name = parts[-2] if len(parts) >= 2 else "unknown"

    for table_name, rel_type in type_map.items():
        relation_rows.append({
            "domain": domain,
            "case": case_name,
            "table": str(table_name),
            "relation_type": str(rel_type).strip(),
            "meta_path": str(meta_path),
        })

print("\n=== 原始关系类型总计 ===")
print(all_type_counter)

relation_type_df = pd.DataFrame(relation_rows)
if relation_type_df.empty:
    print("未找到可用的关系类型数据")
else:
    def normalize_relation_type_list(raw_value):
        """
        支持 relation_type 为：
        - 字符串: "1:2" / "2-hop"
        - 列表: ["1:2", "2-hop"]
        - 其他可序列化类型
        """
        if isinstance(raw_value, list):
            vals = raw_value
        else:
            text = str(raw_value).strip()
            if text.startswith("[") and text.endswith("]"):
                try:
                    parsed = json.loads(text.replace("'", '"'))
                    vals = parsed if isinstance(parsed, list) else [text]
                except Exception:
                    vals = [text]
            else:
                vals = [text]

        out = []
        for v in vals:
            s = str(v).strip().lower()
            if s:
                out.append(s)
        return out

    def map_relation_bucket(type_token: str):
        s = str(type_token).strip().lower()
        if s in {"1:1", "one-to-one", "o2o"}:
            return "1:1"
        if s in {"2-hop", "multi-hop", "multihop", "hop2", "2hop"}:
            return "multi-hop"
        # 将 1:2 / 1:n / n:1 / m:n 等统一归入 1:N
        if ":" in s and s != "1:1":
            return "1:N"
        if s in {"1:n", "n:1", "m:n", "many-to-many", "one-to-many", "many-to-one"}:
            return "1:N"
        return "other"

    relation_type_df = relation_type_df.copy()
    relation_type_df["relation_type_tokens"] = relation_type_df["relation_type"].map(normalize_relation_type_list)
    relation_type_df["relation_buckets"] = relation_type_df["relation_type_tokens"].map(
        lambda xs: sorted({map_relation_bucket(x) for x in xs})
    )

    exploded_bucket_df = relation_type_df.explode("relation_buckets", ignore_index=True)

    print("\n=== 各原始关系类型数量 ===")
    display(
        relation_type_df.groupby("relation_type", as_index=False)
        .size()
        .rename(columns={"size": "count"})
        .sort_values("count", ascending=False)
        .reset_index(drop=True)
    )

    bucket_count_df = (
        exploded_bucket_df.groupby("relation_buckets", as_index=False)
        .size()
        .rename(columns={"relation_buckets": "bucket", "size": "count"})
    )

    # 强制展示目标三类，即使计数为 0 也显示
    target_buckets = ["1:1", "1:N", "multi-hop", "other"]
    bucket_count_df = (
        pd.DataFrame({"bucket": target_buckets})
        .merge(bucket_count_df, on="bucket", how="left")
        .fillna({"count": 0})
    )
    bucket_count_df["count"] = bucket_count_df["count"].astype(int)

    print("\n=== 分类后数量（1:1 / 1:N / multi-hop，可重复计数）===")
    display(bucket_count_df)
    print("分类计数字典:", dict(zip(bucket_count_df["bucket"], bucket_count_df["count"])))

    print("\n=== 关系表明细（含分类，前50）===")
    display(
        relation_type_df[["domain", "case", "table", "relation_type", "relation_buckets", "meta_path"]]
        .sort_values(["domain", "case", "table"])
        .head(50)
        .reset_index(drop=True)
    )

In [ ]:
def normalize_label(label: str) -> str:
    """统一 label 写法，例如 TA_FC 和 TA-FC 归一成 TA-FC。"""
    return str(label).strip().upper().replace("_", "-")


def normalize_table_name(table_name: str) -> str:
    return str(table_name).strip().lower()


def is_non_empty(value) -> bool:
    return value is not None and value != ""


def load_expected_answer(answer_dir: Path):
    expected = {}
    for f in sorted(answer_dir.glob("*.json")):
        with open(f, "r", encoding="utf-8") as fp:
            expected[f.stem] = json.load(fp)
    return expected


def load_label_tables(label_source_path: Path):
    """
    支持两种 label 来源：
    1) meta.json（推荐）：顶层包含 capability_assignments -> {table: {row_key: {...}}}
    2) legacy tables 目录：每张表一个 json 文件
    """
    label_tables = {}

    if label_source_path.is_file():
        with open(label_source_path, "r", encoding="utf-8") as fp:
            meta = json.load(fp)
        assignments_by_table = meta.get("capability_assignments", {}) or {}
        for table_name, table_assignments in assignments_by_table.items():
            label_tables[normalize_table_name(table_name)] = {
                "capability_assignments": table_assignments or {}
            }
        return label_tables

    for f in sorted(label_source_path.glob("*.json")):
        with open(f, "r", encoding="utf-8") as fp:
            data = json.load(fp)
        label_tables[normalize_table_name(f.stem)] = data
    return label_tables


def detect_id_field(row: dict):
    if not isinstance(row, dict) or not row:
        return None
    keys = list(row.keys())
    for k in keys:
        kl = k.lower()
        if kl == "id" or kl.endswith("_id"):
            return k
    return None


def _normalize_assignment_key_text(key: str) -> str:
    return ",".join([seg.strip() for seg in str(key).split(",")])


def _build_row_key_candidates(row: dict, row_idx: int):
    """
    构造用于匹配 meta capability_assignments 的候选 key：
    - 单主键：episode_id / person_id / xxx_id
    - 复合键：如 "episode_id, person_id"（兼容逗号空格差异）
    - 回退：1-based 行号
    """
    candidates = []

    id_field = detect_id_field(row)
    if id_field is not None and row.get(id_field) is not None:
        candidates.append(str(row.get(id_field)))

    fk_like_fields = [k for k in row.keys() if str(k).lower().endswith("_id") and row.get(k) is not None]
    if len(fk_like_fields) >= 2:
        # 按原字段顺序
        candidates.append(", ".join([str(row.get(k)) for k in fk_like_fields]))
        # 按字段名排序，增强鲁棒性
        candidates.append(", ".join([str(row.get(k)) for k in sorted(fk_like_fields, key=lambda x: str(x).lower())]))

    candidates.append(str(row_idx + 1))

    # 去重并保留顺序
    dedup = []
    seen = set()
    for c in candidates:
        k = str(c)
        if k not in seen:
            dedup.append(k)
            seen.add(k)
    return dedup


def get_assignment_entry(table_payload: dict, row_key_candidates):
    assignments = table_payload.get("capability_assignments", {})
    if not isinstance(assignments, dict):
        return {}

    # 构建“归一化key -> 原始key”索引，兼容 'a,b' vs 'a, b'
    normalized_to_raw = {}
    for raw_key in assignments.keys():
        normalized_to_raw[_normalize_assignment_key_text(raw_key)] = raw_key

    for row_key in row_key_candidates:
        if row_key in assignments:
            return assignments[row_key]

        normalized = _normalize_assignment_key_text(row_key)
        if normalized in normalized_to_raw:
            return assignments[normalized_to_raw[normalized]]

        # fallback：有些数据可能直接用 int key 语义
        try:
            int_key = str(int(str(row_key)))
            if int_key in assignments:
                return assignments[int_key]
        except Exception:
            pass

    return {}


def build_capability_lookup(expected_data: dict, label_tables: dict):
    """
    返回两个映射：
    - row_caps[(table, row_idx)] = set(labels)
    - field_caps[(table, row_idx, field)] = set(labels)
    """
    row_caps = defaultdict(set)
    field_caps = defaultdict(set)

    for table_name, expected_rows in expected_data.items():
        tkey = normalize_table_name(table_name)
        payload = label_tables.get(tkey)
        if payload is None or not isinstance(expected_rows, list):
            continue

        for row_idx, row in enumerate(expected_rows):
            if not isinstance(row, dict):
                continue

            row_key_candidates = _build_row_key_candidates(row, row_idx)
            entry = get_assignment_entry(payload, row_key_candidates)
            if not isinstance(entry, dict):
                continue

            # 结构1：{"row": [...], "col": {field: [...]}}
            if "row" in entry or "col" in entry:
                for lb in entry.get("row", []) or []:
                    row_caps[(tkey, row_idx)].add(normalize_label(lb))
                for field, labels in (entry.get("col", {}) or {}).items():
                    for lb in labels or []:
                        field_caps[(tkey, row_idx, str(field).lower())].add(normalize_label(lb))
            else:
                # 结构2：{field: [...]}（无 row/col 包裹）
                for field, labels in entry.items():
                    for lb in labels or []:
                        field_caps[(tkey, row_idx, str(field).lower())].add(normalize_label(lb))

    return row_caps, field_caps


def get_cell_capabilities(table_name: str, row_idx: int, field_name: str, row_caps, field_caps):
    tkey = normalize_table_name(table_name)
    fkey = str(field_name).lower()
    labels = set()
    labels.update(row_caps.get((tkey, row_idx), set()))
    labels.update(field_caps.get((tkey, row_idx, fkey), set()))
    return labels


def aggregate_capability_metrics(comparisons, row_caps, field_caps, include_unlabeled=False):
    stats = defaultdict(lambda: {"expected": 0, "generated": 0, "correct": 0})

    for comp in comparisons:
        table = comp.get("table", "")
        row_idx = comp.get("row_index")
        field = comp.get("field", "")

        labels = get_cell_capabilities(table, row_idx, field, row_caps, field_caps)
        if not labels:
            if not include_unlabeled:
                continue
            labels = {"UNLABELED"}

        expected_non_empty = is_non_empty(comp.get("expected_value"))
        generated_non_empty = is_non_empty(comp.get("generated_value"))
        is_correct = bool(comp.get("match", False)) and expected_non_empty

        for lb in labels:
            if expected_non_empty:
                stats[lb]["expected"] += 1
            if generated_non_empty:
                stats[lb]["generated"] += 1
            if is_correct:
                stats[lb]["correct"] += 1

    rows = []
    for lb, s in stats.items():
        expected = s["expected"]
        generated = s["generated"]
        correct = s["correct"]
        recall = (correct / expected * 100.0) if expected else 0.0
        precision = (correct / generated * 100.0) if generated else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
        rows.append(
            {
                "capability": lb,
                "expected_cells": expected,
                "generated_cells": generated,
                "correct_cells": correct,
                "precision": precision,
                "recall": recall,
                "f1": f1,
            }
        )

    df = pd.DataFrame(rows)
    if df.empty:
        return df
    return df.sort_values(["expected_cells", "f1", "capability"], ascending=[False, False, True]).reset_index(drop=True)


In [ ]:
# ====== capability规则（来自 data_construction/src/capabilities.py） ======
PILLAR1_KEYS = {
    "TA_FC", "TA_US", "TA_EM",
    "RI_AC", "RI_LD", "RI_TC", "RI_MI",
    "TD_AS", "TD_DF", "TD_CA",
    "EF_ND", "EF_HC", "EF_RC",
}

PILLAR2_KEYS = {
    "RL_O2M", "RL_MB", "RL_CL",
    "IDR_ED", "IDR_CR",
    "GR_TI", "RL_MI", "GR_DC",
    "IC_ME", "IC_NR",
}

# 规则（来自 statitic.ipynb）
IGNORED_CAPABILITIES = {"IC_RI"}
CAPABILITY_ALIASES = {
    "GR_CTA": "RL_MI",
}


def normalize_capability_key(label: str):
    """统一为下划线风格，并应用忽略和别名映射。"""
    key = str(label).strip().upper().replace("-", "_")
    if key in IGNORED_CAPABILITIES:
        return None
    key = CAPABILITY_ALIASES.get(key, key)
    return key


PILLAR1_KEYS_NORM = {k for k in (normalize_capability_key(x) for x in PILLAR1_KEYS) if k}
PILLAR2_KEYS_NORM = {k for k in (normalize_capability_key(x) for x in PILLAR2_KEYS) if k}


def get_relation_tables_from_meta(label_source_path: Path):
    """
    从 meta.json 顶层 type 读取关系表。
    例如: {"Award": "1:2", "Credit": "1:2", "Vote": "1:1"}
    """
    if not label_source_path.is_file():
        return set()

    with open(label_source_path, "r", encoding="utf-8") as fp:
        meta = json.load(fp)

    relation_type_map = meta.get("type", {}) or {}
    if not isinstance(relation_type_map, dict):
        return set()

    return {normalize_table_name(t) for t in relation_type_map.keys()}


def get_normalized_cell_capabilities(table_name: str, row_idx: int, field_name: str, row_caps, field_caps):
    raw_labels = get_cell_capabilities(table_name, row_idx, field_name, row_caps, field_caps)
    normalized = set()
    for lb in raw_labels:
        mapped = normalize_capability_key(lb)
        if mapped:
            normalized.add(mapped)
    return normalized


def load_schema(schema_path: Path):
    with open(schema_path, "r", encoding="utf-8") as fp:
        return json.load(fp)


def get_relation_fk_fields_from_schema(schema):
    """返回 {relation_table_lower: set(fk_field_lower)}。"""
    relation_fk_fields = defaultdict(set)
    if not isinstance(schema, list):
        return relation_fk_fields

    for table in schema:
        if not isinstance(table, dict):
            continue
        table_name = normalize_table_name(table.get("name", ""))
        table_type = str(table.get("type", "")).lower()
        if table_type != "relation":
            continue

        fields = table.get("fields", []) or table.get("attributes", [])
        for field in fields:
            if not isinstance(field, dict):
                continue
            field_name = str(field.get("name", "")).lower()
            constraints = field.get("constraints", {})
            if not field_name or not isinstance(constraints, dict):
                continue
            if "foreign_key" in constraints:
                relation_fk_fields[table_name].add(field_name)

    return relation_fk_fields


def _row_fk_comparisons(row_comparisons, fk_fields):
    fk_fields = {str(f).lower() for f in (fk_fields or set())}
    return [
        c for c in row_comparisons
        if str(c.get("field", "")).lower() in fk_fields
    ]


def _row_is_generated_by_fk(row_comparisons, fk_fields):
    fk_comps = _row_fk_comparisons(row_comparisons, fk_fields)
    if not fk_comps:
        return False
    return any(is_non_empty(c.get("generated_value")) for c in fk_comps)


def _row_is_correct_by_fk(row_comparisons, fk_fields):
    # 行级正确：该行所有“外键且 expected 非空”的cell都匹配成功
    fk_comps = _row_fk_comparisons(row_comparisons, fk_fields)
    expected_fk_cells = [c for c in fk_comps if is_non_empty(c.get("expected_value"))]
    if not expected_fk_cells:
        return False
    return all(bool(c.get("match", False)) for c in expected_fk_cells)


def aggregate_capability_metrics(
    comparisons,
    row_caps,
    field_caps,
    relation_tables=None,
    relation_fk_fields_by_table=None,
    include_unlabeled=False,
):
    """
    输出：
    1) capability_df：每个capability的precision/recall/f1
    2) pillar_summary_df：pillar1/pillar2汇总分

    统计口径：
    - pillar1：按cell统计（实体内能力）
    - pillar2：按关系表行统计，且仅按schema外键字段判定行正确
    """
    relation_tables = {normalize_table_name(t) for t in (relation_tables or set())}
    relation_fk_fields_by_table = {
        normalize_table_name(k): {str(x).lower() for x in v}
        for k, v in (relation_fk_fields_by_table or {}).items()
    }

    stats = defaultdict(lambda: {
        "expected": 0,
        "generated": 0,
        "correct": 0,
        "pillar": "unknown",
        "granularity": "unknown",
    })

    row_group = defaultdict(list)
    for comp in comparisons:
        table = normalize_table_name(comp.get("table", ""))
        row_idx = comp.get("row_index")
        row_group[(table, row_idx)].append(comp)

    # ===== pillar1: cell-level =====
    for comp in comparisons:
        table = comp.get("table", "")
        row_idx = comp.get("row_index")
        field = comp.get("field", "")

        labels = get_normalized_cell_capabilities(table, row_idx, field, row_caps, field_caps)
        labels = {lb for lb in labels if lb in PILLAR1_KEYS_NORM}

        if not labels:
            if not include_unlabeled:
                continue
            labels = {"UNLABELED_P1"}

        expected_non_empty = is_non_empty(comp.get("expected_value"))
        generated_non_empty = is_non_empty(comp.get("generated_value"))
        is_correct = bool(comp.get("match", False)) and expected_non_empty

        for lb in labels:
            stats[lb]["pillar"] = "pillar1"
            stats[lb]["granularity"] = "cell"
            if expected_non_empty:
                stats[lb]["expected"] += 1
            if generated_non_empty:
                stats[lb]["generated"] += 1
            if is_correct:
                stats[lb]["correct"] += 1

    # ===== pillar2: row-level on relation tables (FK-based) =====
    for (table, row_idx), row_comparisons in row_group.items():
        if relation_tables and table not in relation_tables:
            continue

        row_labels = set()
        for comp in row_comparisons:
            row_labels.update(
                get_normalized_cell_capabilities(
                    comp.get("table", ""),
                    comp.get("row_index"),
                    comp.get("field", ""),
                    row_caps,
                    field_caps,
                )
            )

        row_labels = {lb for lb in row_labels if lb in PILLAR2_KEYS_NORM}
        if not row_labels:
            if not include_unlabeled:
                continue
            row_labels = {"UNLABELED_P2"}

        fk_fields = relation_fk_fields_by_table.get(table, set())
        row_generated = _row_is_generated_by_fk(row_comparisons, fk_fields)
        row_correct = _row_is_correct_by_fk(row_comparisons, fk_fields)

        for lb in row_labels:
            stats[lb]["pillar"] = "pillar2"
            stats[lb]["granularity"] = "row"
            stats[lb]["expected"] += 1
            if row_generated:
                stats[lb]["generated"] += 1
            if row_correct:
                stats[lb]["correct"] += 1

    rows = []
    for lb, s in stats.items():
        expected = s["expected"]
        generated = s["generated"]
        correct = s["correct"]
        recall = (correct / expected * 100.0) if expected else 0.0
        precision = (correct / generated * 100.0) if generated else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
        rows.append(
            {
                "capability": lb,
                "pillar": s["pillar"],
                "granularity": s["granularity"],
                "expected_units": expected,
                "generated_units": generated,
                "correct_units": correct,
                "precision": precision,
                "recall": recall,
                "f1": f1,
            }
        )

    capability_df = pd.DataFrame(rows)
    if capability_df.empty:
        return capability_df, pd.DataFrame()

    capability_df = capability_df.sort_values(
        ["pillar", "expected_units", "f1", "capability"],
        ascending=[True, False, False, True],
    ).reset_index(drop=True)

    pillar_rows = []
    for pillar_name in ["pillar1", "pillar2"]:
        sub = capability_df[capability_df["pillar"] == pillar_name]
        expected = int(sub["expected_units"].sum()) if not sub.empty else 0
        generated = int(sub["generated_units"].sum()) if not sub.empty else 0
        correct = int(sub["correct_units"].sum()) if not sub.empty else 0
        recall = (correct / expected * 100.0) if expected else 0.0
        precision = (correct / generated * 100.0) if generated else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
        pillar_rows.append(
            {
                "pillar": pillar_name,
                "expected_units": expected,
                "generated_units": generated,
                "correct_units": correct,
                "precision": precision,
                "recall": recall,
                "f1": f1,
            }
        )

    pillar_summary_df = pd.DataFrame(pillar_rows)
    return capability_df, pillar_summary_df


def build_label_presence_comparison(capability_df: pd.DataFrame):
    """
    统计每个pillar下：有标签 vs 无标签 的准确率（correct/expected）。
    """
    if capability_df is None or capability_df.empty:
        return pd.DataFrame()

    rows = []
    for pillar_name in ["pillar1", "pillar2"]:
        sub = capability_df[capability_df["pillar"] == pillar_name]
        if sub.empty:
            continue

        unlabeled_tag = "UNLABELED_P1" if pillar_name == "pillar1" else "UNLABELED_P2"

        labeled_sub = sub[sub["capability"] != unlabeled_tag]
        unlabeled_sub = sub[sub["capability"] == unlabeled_tag]

        for group_name, group_df in [("labeled", labeled_sub), ("unlabeled", unlabeled_sub)]:
            expected = int(group_df["expected_units"].sum()) if not group_df.empty else 0
            generated = int(group_df["generated_units"].sum()) if not group_df.empty else 0
            correct = int(group_df["correct_units"].sum()) if not group_df.empty else 0
            accuracy = (correct / expected * 100.0) if expected else 0.0
            precision = (correct / generated * 100.0) if generated else 0.0

            rows.append(
                {
                    "pillar": pillar_name,
                    "label_group": group_name,
                    "expected_units": expected,
                    "generated_units": generated,
                    "correct_units": correct,
                    "accuracy": accuracy,
                    "precision": precision,
                }
            )

    result_df = pd.DataFrame(rows)
    if result_df.empty:
        return result_df
    return result_df.sort_values(["pillar", "label_group"]).reset_index(drop=True)

In [ ]:
# 1) 读取数据
expected_data = load_expected_answer(ANSWER_DIR)
with open(GENERATED_DATA_PATH, "r", encoding="utf-8") as fp:
    generated_data = json.load(fp)
schema = load_schema(SCHEMA_PATH)
label_tables = load_label_tables(LABEL_SOURCE_PATH)
relation_tables = get_relation_tables_from_meta(LABEL_SOURCE_PATH)
relation_fk_fields_by_table = get_relation_fk_fields_from_schema(schema)

# 2) 复用 test_dataset_evaluate.py 中函数，获得逐 cell 比对结果
evaluator = Doc2DBEvaluator(
    backend_url="http://localhost:5000",
    dataset_dir=Path("."),
    output_dir=Path("./_tmp_eval_output"),
    enable_llm_eval=False,
)
case_info = {
    "case_dir": ANSWER_DIR.parent,
}
cell_metrics = evaluator.calculate_cell_metrics(
    generated_data,
    expected_data,
    schema=schema,
    case_info=case_info,
)
comparisons = cell_metrics.get("detailed_comparisons", [])

print("Overall from calculate_cell_metrics:")
print(f"  precision={cell_metrics.get('cell_precision', 0):.2f}  recall={cell_metrics.get('cell_recall', 0):.2f}  f1={cell_metrics.get('cell_f1', 0):.2f}")
print(f"  detailed comparisons={len(comparisons)}")
print(f"relation tables from meta.type = {sorted(relation_tables)}")
print(f"relation fk fields from schema = {dict((k, sorted(v)) for k, v in relation_fk_fields_by_table.items())}")

# 3) 构建 capability 索引并聚合（pillar1/pillar2分开）
row_caps, field_caps = build_capability_lookup(expected_data, label_tables)
capability_df, pillar_summary_df = aggregate_capability_metrics(
    comparisons,
    row_caps,
    field_caps,
    relation_tables=relation_tables,
    relation_fk_fields_by_table=relation_fk_fields_by_table,
    include_unlabeled=True,
)
label_presence_df = build_label_presence_comparison(capability_df)

print("\nPillar summary:")
display(pillar_summary_df)

print("\n有标签 vs 无标签 准确率对比（accuracy=correct/expected）:")
label_presence_df


In [ ]:
# 4) 调试：打印 pillar2 行级明细（FK判定）
def debug_pillar2_rows(comparisons, row_caps, field_caps, relation_tables, relation_fk_fields_by_table):
    relation_tables = {normalize_table_name(t) for t in relation_tables}
    relation_fk_fields_by_table = {
        normalize_table_name(k): {str(x).lower() for x in v}
        for k, v in (relation_fk_fields_by_table or {}).items()
    }

    row_group = defaultdict(list)
    for comp in comparisons:
        table = normalize_table_name(comp.get("table", ""))
        row_idx = comp.get("row_index")
        row_group[(table, row_idx)].append(comp)

    debug_rows = []
    for (table, row_idx), row_comparisons in sorted(row_group.items(), key=lambda x: (x[0][0], x[0][1])):
        if relation_tables and table not in relation_tables:
            continue

        row_labels = set()
        for comp in row_comparisons:
            row_labels.update(
                get_normalized_cell_capabilities(
                    comp.get("table", ""),
                    comp.get("row_index"),
                    comp.get("field", ""),
                    row_caps,
                    field_caps,
                )
            )
        row_labels = {lb for lb in row_labels if lb in PILLAR2_KEYS_NORM}
        if not row_labels:
            continue

        fk_fields = relation_fk_fields_by_table.get(table, set())
        fk_comparisons = _row_fk_comparisons(row_comparisons, fk_fields)
        row_generated = _row_is_generated_by_fk(row_comparisons, fk_fields)
        row_correct = _row_is_correct_by_fk(row_comparisons, fk_fields)

        fk_failed_fields = []
        for c in fk_comparisons:
            if is_non_empty(c.get("expected_value")) and not bool(c.get("match", False)):
                fk_failed_fields.append(
                    {
                        "field": c.get("field"),
                        "expected": c.get("expected_value"),
                        "generated": c.get("generated_value"),
                    }
                )

        debug_rows.append(
            {
                "table": table,
                "row_index": row_idx,
                "labels": ",".join(sorted(row_labels)),
                "fk_fields": ",".join(sorted(fk_fields)),
                "row_generated_by_fk": row_generated,
                "row_correct_by_fk": row_correct,
                "fk_failed_field_count": len(fk_failed_fields),
                "fk_failed_fields_preview": fk_failed_fields[:3],
            }
        )

    df = pd.DataFrame(debug_rows)
    if df.empty:
        print("没有命中 pillar2 的关系行。")
        return df

    df = df.sort_values(["row_correct_by_fk", "table", "row_index"], ascending=[False, True, True]).reset_index(drop=True)

    print(f"pillar2 行总数: {len(df)}")
    print(f"pillar2 按FK正确行数: {int(df['row_correct_by_fk'].sum())}")

    print("\n=== 按FK正确的行 ===")
    display(df[df["row_correct_by_fk"] == True][["table", "row_index", "labels", "fk_fields", "fk_failed_field_count"]])

    print("\n=== 按FK错误的行（前50）===")
    display(df[df["row_correct_by_fk"] == False].head(50))

    return df


pillar2_debug_df = debug_pillar2_rows(
    comparisons,
    row_caps,
    field_caps,
    relation_tables,
    relation_fk_fields_by_table,
)

In [ ]:
# 可选：保存结果到 CSV
out_csv = Path("./capability_metrics.csv")
capability_df.to_csv(out_csv, index=False, encoding="utf-8")
print(f"saved: {out_csv.resolve()}")


In [ ]:
import json
import pandas as pd
from pathlib import Path
from collections import defaultdict, Counter
from test_dataset_evaluate import Doc2DBEvaluator

# ====== 1. 路径与模型配置 ======
BASE_CASE_DIR = Path("dataset/case/base_latest")
LATEST_OUTPUT_DIR = Path("dataset/case/base_latest_output/llm")
OUT_DIR = Path("./_folder_analysis_output")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ====== 2. Pillar 定义与归一化逻辑 ======
PILLAR1_KEYS = {"TA_FC", "TA_US", "TA_EM", "RI_AC", "RI_LD", "RI_TC", "RI_MI", "TD_AS", "TD_DF", "TD_CA", "EF_ND", "EF_HC", "EF_RC"}
PILLAR2_KEYS = {"RL_O2M", "RL_MB", "RL_CL", "IDR_ED", "IDR_CR", "GR_TI", "RL_MI", "GR_DC", "IC_ME", "IC_NR"}
CAPABILITY_ALIASES = {"GR_CTA": "RL_MI"}

def normalize_label(label: str) -> str:
    return str(label).strip().upper().replace("_", "-")

def normalize_capability_key(label: str):
    key = str(label).strip().upper().replace("-", "_")
    return CAPABILITY_ALIASES.get(key, key)

PILLAR1_KEYS_NORM = {normalize_capability_key(k) for k in PILLAR1_KEYS}
PILLAR2_KEYS_NORM = {normalize_capability_key(k) for k in PILLAR2_KEYS}

# ====== 3. 核心评估函数 (封装单文件处理) ======
def evaluate_single_file(generated_path: Path, evaluator: Doc2DBEvaluator):
    # 解析路径获取元数据
    rel_parts = generated_path.relative_to(LATEST_OUTPUT_DIR).parts
    domain, case_name = rel_parts[0], rel_parts[1]
    
    case_dir = BASE_CASE_DIR / domain / case_name
    answer_dir = case_dir / "tables"
    meta_path = case_dir / "meta.json"
    schema_path = case_dir / "schema.json"

    # 加载基础数据
    with open(generated_path, "r") as f: gen_data = json.load(f)
    with open(schema_path, "r") as f: schema = json.load(f)
    expected_data = {}
    for f in sorted(answer_dir.glob("*.json")):
        with open(f, "r") as fp: expected_data[f.stem] = json.load(fp)

    # 运行基础 Cell 评估
    cell_metrics = evaluator.calculate_cell_metrics(gen_data, expected_data, schema=schema)
    comparisons = cell_metrics.get("detailed_comparisons", [])

    # 获取标签映射
    label_tables = load_label_tables(meta_path) # 复用你原有的 load_label_tables
    row_caps, field_caps = build_capability_lookup(expected_data, label_tables)
    
    # 聚合 Pillar 指标
    cap_df, pillar_df = aggregate_capability_metrics(
        comparisons, row_caps, field_caps, 
        relation_tables=get_relation_tables_from_meta(meta_path),
        relation_fk_fields_by_table=get_relation_fk_fields_from_schema(schema),
        include_unlabeled=True
    )
    
    # 计算有无标签对比
    presence_df = build_label_presence_comparison(cap_df)
    
    return cap_df, pillar_df, presence_df

# ====== 4. 批量执行与结果汇总 ======
# evaluator = Doc2DBEvaluator(enable_llm_eval=False)
evaluator = Doc2DBEvaluator(
    backend_url="http://localhost:5000",
    dataset_dir=Path("."),
    output_dir=Path("./_tmp_eval_output"),
    enable_llm_eval=False,
)
all_pillar_results = []
all_presence_results = []
all_cap_results = []      # 用于存储细粒度 Label 数据

from pathlib import Path

# 确保你的基础路径是正确的
print(f"当前设置的 LATEST_OUTPUT_DIR: {LATEST_OUTPUT_DIR}")

TARGET_MODEL = "qwen3-max"
model_key = TARGET_MODEL.lower().replace('-', '_') if TARGET_MODEL else None
print(f"当前过滤的模型关键字: '{model_key}'\n")


files_to_eval = []
model_key = TARGET_MODEL.lower().replace('-', '_') if TARGET_MODEL else None

print(f"正在针对模型关键字 '{model_key}' 扫描所有测试案卷...")

# 遍历 domain/case/run_name
for run_dir in LATEST_OUTPUT_DIR.glob("*/*/*"):
    if not run_dir.is_dir():
        continue
    
    # 归一化目录名进行匹配
    dir_name = run_dir.name.lower().replace('-', '_')
    
    # 核心修改：如果指定了 target_model，则只看匹配的目录
    if model_key and model_key not in dir_name:
        continue
        
    # 针对该模型目录，寻找最合适的数据文件
    # 优先级：*extracted_data_pipeline_oracle.json > *extracted_data.json
    target_file = None
    
    # 尝试寻找 Oracle 文件
    oracles = list(run_dir.glob("*extracted_data_pipeline_oracle.json")) or \
              list(run_dir.glob("extracted_data_pipeline_oracle.json"))
    
    if oracles:
        target_file = oracles[0]
    else:
        # 尝试寻找基础 Extracted 文件
        bases = list(run_dir.glob("*extracted_data.json")) or \
                list(run_dir.glob("extracted_data.json"))
        if bases:
            target_file = bases[0]
            
    if target_file:
        files_to_eval.append(target_file)

print(f"扫描完毕，成功找到模型 '{TARGET_MODEL}' 的 {len(files_to_eval)} 个可评估案例。")

for p in files_to_eval:
    try:
        # 解包 evaluate_single_file 返回的 3 个值
        cap_df, pillar_df, presence_df = evaluate_single_file(p, evaluator)
        
        if not cap_df.empty: 
            all_cap_results.append(cap_df) # 收集细粒度数据
        if not pillar_df.empty: 
            all_pillar_results.append(pillar_df)
        if not presence_df.empty: 
            all_presence_results.append(presence_df)
            
    except Exception as e:
        print(f"跳过失败案例 {p}: {e}")

# ====== 5. 输出汇总报告 ======
# A. 每个细粒度 Label 的 F1 Score 统计 [新增]
if all_cap_results:
    # 合并所有案卷的细粒度数据
    full_cap_df = pd.concat(all_cap_results)
    
    # 按 capability (label) 分组，同时保留它所属的 pillar 属性
    # 注意：这里使用 mean 是做 Macro Average (每个 case 的表现平均)
    label_summary = full_cap_df.groupby(["pillar", "capability"]).agg({
        "precision": "mean",
        "recall": "mean",
        "f1": "mean",
        "expected_units": "sum" # 统计该标签在测试集中出现的总频次
    }).reset_index()

    # 按 Pillar 和 F1 降序排序
    label_summary = label_summary.sort_values(["pillar", "f1"], ascending=[True, False])

    print(f"\n=== 模型 {TARGET_MODEL}：细粒度 Label 性能统计 (Macro Average) ===")
    display(label_summary)
    
    # 保存 Label 细粒度结果
    label_summary.to_csv(OUT_DIR / f"{TARGET_MODEL}_label_detailed_metrics.csv", index=False)

    # A2. 一级分类（如 TA / TD / IDR / RL / UNLABELED）统计
    def capability_top_category(cap_name: str) -> str:
        name = str(cap_name).strip().upper()
        if name.startswith("UNLABELED"):
            return "UNLABELED"
        if "_" in name:
            return name.split("_", 1)[0]
        if "-" in name:
            return name.split("-", 1)[0]
        return name

    label_summary_with_cat = label_summary.copy()
    label_summary_with_cat["category"] = label_summary_with_cat["capability"].map(capability_top_category)

    category_summary = (
        label_summary_with_cat.groupby(["pillar", "category"], as_index=False)
        .agg({
            "precision": "mean",
            "recall": "mean",
            "f1": "mean",
            "expected_units": "sum",
        })
        .sort_values(["pillar", "f1", "expected_units"], ascending=[True, False, False])
        .reset_index(drop=True)
    )

    print(f"\n=== 模型 {TARGET_MODEL}：一级分类性能统计 (TA/TD/IDR...) ===")
    display(category_summary)

    category_summary.to_csv(OUT_DIR / f"{TARGET_MODEL}_category_metrics.csv", index=False)

# B. 各 Pillar 性能汇总
if all_pillar_results:
    final_pillar = pd.concat(all_pillar_results).groupby("pillar").mean(numeric_only=True)
    print("\n=== 整个文件夹下各 Pillar 性能汇总 (Macro Average) ===")
    display(final_pillar[["precision", "recall", "f1"]])
    final_pillar.to_csv(OUT_DIR / "folder_pillar_summary.csv")

# C. 有无标签效果对比 (Presence Analysis)
if all_presence_results:
    # 确保在 display 中加入 f1 分数
    final_presence = pd.concat(all_presence_results).groupby(["pillar", "label_group"]).mean(numeric_only=True)
    print("\n=== 有无标签效果对比 (Presence Analysis) ===")
    display(final_presence[["accuracy", "precision"]])

    # 保存结果
    final_presence.to_csv(OUT_DIR / "folder_label_presence.csv")